# AlphaFold2 Structural Features — Colab (CPU-only)

> **يشتغل على CPU runtime — ما يحتاج GPU.**  يقدر يشتغل بالتوازي مع notebook ESM-2 اللي يستخدم الـGPU.

## ماذا يعمل؟

يستخرج ٤ structural features لكل missense variant:
- `pLDDT_position` — AlphaFold2 confidence في موقع الـvariant
- `pLDDT_window_5` — متوسط الـ confidence في نافذة ±٥ bacteria حول الموقع
- `SASA_position` — Solvent Accessible Surface Area بالـ Å²
- `neighbors_5A` — عدد المجاورات في نطاق 5Å

## المدخلات

- `data/intermediate/esm2/vep_ann.parquet` (من Drive — كتبه notebook 11)
- `data/splits/{train,val,test}.parquet` (من Drive)

## المخرجات

- `data/intermediate/alphafold/features.parquet` (Drive cache)
- Push للـGitHub تلقائياً في آخر خلية

## الإعداد (١٠ ثواني)

1. **Runtime → Change runtime type → CPU** (لا GPU!)
2. **Runtime → Run all**

بس! AlphaFold pipeline سريع لأنه شغل CPU بسيط (biopython + freesasa) + HTTP downloads.

## ١. تثبيت المتطلبات

In [ ]:
%pip install -q biopython freesasa pandas pyarrow requests
import biopython  # noqa
print('biopython + freesasa ready')

## ٢. ربط Google Drive + استنساخ الـRepo

الـnotebook 11 (ESM-2) حفظ الـ VEP cache في `/content/drive/MyDrive/missense_esm2/cache/`. نستخدم نفس الـ Drive هنا.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

GH_PAT = userdata.get('GH_PAT')
REPO = 'RayanAlDwlah/Genetic-Mutation-Detection-project'
!rm -rf /content/repo && git clone https://{GH_PAT}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git log --oneline -3

## ٣. ربط الـ ESM-2 Drive cache

`vep_ann.parquet` محفوظ في Drive من notebook 11. نعمل symlink حتى يشوفه الـ pipeline.

In [ ]:
import os

ESM2_CACHE = '/content/drive/MyDrive/missense_esm2/cache'
AF_CACHE = '/content/drive/MyDrive/missense_alphafold/cache'
os.makedirs(AF_CACHE, exist_ok=True)

# symlink ESM-2 cache (contains vep_ann.parquet with protein_position)
local_esm = '/content/repo/data/intermediate/esm2'
if os.path.lexists(local_esm):
    os.remove(local_esm) if os.path.islink(local_esm) else __import__('shutil').rmtree(local_esm)
os.makedirs('/content/repo/data/intermediate', exist_ok=True)
os.symlink(ESM2_CACHE, local_esm)

# symlink alphafold cache (will hold our outputs)
local_af = '/content/repo/data/intermediate/alphafold'
if os.path.lexists(local_af):
    os.remove(local_af) if os.path.islink(local_af) else __import__('shutil').rmtree(local_af)
os.symlink(AF_CACHE, local_af)

!ls -la /content/repo/data/intermediate/
!ls -la /content/drive/MyDrive/missense_esm2/cache/ | head

## ٤. تشخيص: كم variant جاهز للمعالجة؟

نحتاج variants فيها `protein_position` (من VEP) + gene (من splits). إذا لسّا ESM-2 train يكمّل، نشتغل فقط على val + test.

In [ ]:
import pandas as pd

vep = pd.read_parquet('/content/repo/data/intermediate/esm2/vep_ann.parquet')
print(f'VEP cache: {len(vep):,} variants, {vep["protein_position"].notna().sum():,} with protein_position')

splits = []
for s in ('train', 'val', 'test'):
    df = pd.read_parquet(f'/content/repo/data/splits/{s}.parquet')[['variant_key', 'gene']]
    splits.append(df.assign(split=s))
splits_df = pd.concat(splits, ignore_index=True).drop_duplicates('variant_key')

merged = vep.merge(splits_df, on='variant_key', how='inner').dropna(subset=['gene', 'protein_position'])
print(f'\nVariants with gene + protein_position:')
print(merged.groupby('split').size().to_string())
print(f'\nTotal: {len(merged):,}')

## ٥. تشغيل AlphaFold feature extraction

الـ pipeline في `scripts/run_alphafold_features.py`. نشغّله مباشرة.

**الوقت المتوقع**: ~2–4 ساعات لكامل val + test + train (إذا train جاهز).

Checkpointing كل 200 variant → لو انقطعت الجلسة نكمّل من Drive بدون ما نضيّع شي.

In [ ]:
import sys
sys.path.insert(0, '/content/repo')

!cd /content/repo && PYTHONPATH=. python scripts/run_alphafold_features.py \
    --vep-cache data/intermediate/esm2/vep_ann.parquet \
    --out data/intermediate/alphafold/features.parquet \
    --splits-dir data/splits

## ٦. Push النتايج على GitHub

In [ ]:
%cd /content/repo
!git config user.email 'colab@automation'
!git config user.name 'Colab Automation'
# نسخ features parquet من الـ symlink إلى path عادي عشان git يشوفها
!cp /content/drive/MyDrive/missense_alphafold/cache/features.parquet data/intermediate/alphafold/features.parquet 2>/dev/null || true
!cp /content/drive/MyDrive/missense_alphafold/cache/gene_to_uniprot.json data/intermediate/alphafold/ 2>/dev/null || true
!cp /content/drive/MyDrive/missense_alphafold/cache/uniprot_to_pdb_url.json data/intermediate/alphafold/ 2>/dev/null || true
!git add -f data/intermediate/alphafold/features.parquet data/intermediate/alphafold/*.json
!git commit -m 'Stage 2.2 AlphaFold: pLDDT + SASA + neighbors features (Colab)'
!git push origin HEAD:main

## ٧. خلاص!

لما تشوف الرسالة `[alphafold] summary: X/Y variants successfully scored`، كل شي تمام.

على الجهاز المحلي، أنا راح أسحب هذي الـ parquet وأدمجها مع ESM-2 لإعادة تدريب XGBoost.